# W02c - Topic Modeling & Predictions with ML Regression

In [1]:
### Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import random
import gensim
import pickle
import time
from text_operations import manipulate_text, translate_to_english
from gensim.utils import simple_preprocess
from gensim.test.utils import datapath
from gensim.parsing.preprocessing import STOPWORDS
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_score, cross_validate, GridSearchCV
import xgboost
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns',None)
num_topics = 25

In [2]:
### Load the salary dataset with the correct date, plus the country coefficient dataset
salaries = pd.read_csv('Salaries_v4_202412161100_enhanced_tm.csv')
salaries = salaries.drop('Unnamed: 0', axis=1)
countries_coef = pd.read_csv('countries_salary_coef.csv')

## Dataset Information

In [3]:
### Number of rows (salaries) and columns
print("SALARY DATASET SHAPE -> ROWS: {}  COLUMNS: {}".format(salaries.shape[0], salaries.shape[1]))

SALARY DATASET SHAPE -> ROWS: 19554  COLUMNS: 40


In [4]:
### First few rows of the dataset
salaries.head()

,id,found_country,title,position,employment_type,company,company_score,edu_degrees,edu_degrees_major,working_year,education_score,ms_counts,skill_counts,main_skills,skills,amount_usd,country_coef,title_code,title_slr_coef,title_seniority,title_seniority_slr_coef,position_code,position_slr_coef,position_seniority,position_seniority_slr_coef,employment_type_coef,emp_type,employment_type_slr_coef,highest_edu_deg_name,highest_edu_deg_lvl,edu_deg_slr_coef,edu_degree_major_code,edu_degree_major_slr_coef_avg,edu_degree_major_slr_coef_max,company_code,company_slr_coef,main_skill_code,main_skill_slr_coef_avg,main_skill_slr_coef_max,topic_model_no
0,0000083c-7054-4a2b-b675-6ac664c66bfb,United States,"Software Developer II at Audible, Inc.",Software Developer II,Full-time,"Audible, Inc.",8.9,"HIGH_SCHOOL,MASTERS,UNDERGRADUATE","Bachelor’s Degree, Computer Science,High Schoo...",11,8.1,21,20,"Ajax, C, C++, CSS, Data Engineering, HTML, HTM...","ajax, c, c++, cascading style sheets (css), cs...",192000,1.00,WC68,0.488,NO_SENIORITY,0.488,WC66,0.492,NO_SENIORITY,0.518,0.99,FULL_TIME,1.0,MASTERS,0.75,0.770,EDM17,0.796,0.796,UNCT,0.368,"MS01,MS02,MS03,MS04,MS05,MS06,MS07,MS11,MS17,M...",0.872,0.944,5
1,00013847-ecf1-4a5a-ba44-16475dc28eba,United States,Retail Associate at Converse,Retail Associate,Full-time,NaN,NaN,UNDERGRADUATE,NaN,5,NaN,14,10,"Communication, Customer Service, Deadline Mana...","communication, customer service, employee trai...",38000,1.00,UNCT,0.319,ASSOCIATE,0.319,UNCT,0.329,ASSOCIATE,0.340,0.99,FULL_TIME,1.0,UNDERGRADUATE,0.50,0.657,NO_DEGREE,0.200,0.200,NO_COMPANY,0.200,"MS00,MS13,MS15,MS16,MS28,MS30,MS31,MS33,MS34,M...",0.646,0.731,0
2,00018332-5b5d-4c23-88f8-1c2cdc133e28,United Kingdom,Test Engineer at Sky,Test Engineer,Full-time,NaN,NaN,UNDERGRADUATE,"Bachelor of Science (BSc), Computer Software E...",12,7.0,15,20,"Agile, Attention to Detail, Automated Testing,...","agile methodologies, api testing, communicatio...",66191,0.61,WC76,0.399,NO_SENIORITY,0.488,WC67,0.400,NO_SENIORITY,0.518,0.99,FULL_TIME,1.0,UNDERGRADUATE,0.50,0.657,"EDM14,EDM30,EDM38,EDM53",0.779,0.823,NO_COMPANY,0.200,"MS00,MS16,MS40,MS42,MS43,MS45,MS62",0.774,0.844,21
3,000c1054-ab28-4c4d-90b0-fa4b1ed31a2a,United States,Hardware Engineer at Google,Hardware Engineer,NaN,Google,8.7,MASTERS,"Master of Science (MS), Computer Engineering",13,8.9,9,20,"C, Circuit Design, Debugging, Embedded Softwar...","asic, c, computer architecture, debugging, emb...",341000,1.00,WC36,0.955,NO_SENIORITY,0.488,WC32,0.959,NO_SENIORITY,0.518,0.10,NO_EMP_TYPE,0.2,MASTERS,0.75,0.770,EDM14,0.823,0.823,CMP01,0.845,"MS20,MS42,MS43",0.873,0.944,17
4,00145b03-e286-4bdc-9063-ed5d2095306a,United States,BI Engineer @ Amazon | MS,BIE II,NaN,Amazon,8.4,MASTERS,"Master of Science - MS, Data Analytics",3,8.5,6,9,"JavaScript, Marketing Research, Microsoft Exce...","javascript, microsoft excel, microsoft office,...",185000,1.00,WC07,0.503,NO_SENIORITY,0.488,WC06,0.516,NO_SENIORITY,0.518,0.10,NO_EMP_TYPE,0.2,MASTERS,0.75,0.770,EDM19,0.655,0.655,CMP00,0.593,"MS03,MS08,MS12,MS14,MS19",0.769,0.870,24


In [5]:
### All features' info in the salary dataset
salaries.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19554 entries, 0 to 19553
Data columns (total 40 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   id                             19554 non-null  object 
 1   found_country                  19554 non-null  object 
 2   title                          19542 non-null  object 
 3   position                       19554 non-null  object 
 4   employment_type                12397 non-null  object 
 5   company                        16936 non-null  object 
 6   company_score                  15739 non-null  float64
 7   edu_degrees                    19000 non-null  object 
 8   edu_degrees_major              18623 non-null  object 
 9   working_year                   19554 non-null  int64  
 10  education_score                14956 non-null  float64
 11  ms_counts                      19554 non-null  int64  
 12  skill_counts                   19554 non-null 

In [6]:
### Stats for numerical features of the dataset
salaries.describe()

,company_score,working_year,education_score,ms_counts,skill_counts,amount_usd,country_coef,title_slr_coef,title_seniority_slr_coef,position_slr_coef,position_seniority_slr_coef,employment_type_coef,employment_type_slr_coef,highest_edu_deg_lvl,edu_deg_slr_coef,edu_degree_major_slr_coef_avg,edu_degree_major_slr_coef_max,company_slr_coef,main_skill_slr_coef_avg,main_skill_slr_coef_max,topic_model_no
count,15739.000000,19554.000000,14956.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000
mean,8.248993,11.449882,8.401484,15.738008,18.777948,163366.552981,0.966054,0.452360,0.511637,0.456774,0.550829,0.651904,0.701566,0.612090,0.705434,0.695443,0.714984,0.517684,0.784424,0.884132,12.980158
std,0.561583,7.213469,1.383108,7.306042,7.814769,81118.159433,0.109942,0.145708,0.074651,0.148300,0.087546,0.426533,0.382863,0.178708,0.128333,0.147528,0.158501,0.206434,0.083624,0.089182,7.183227
min,3.800000,0.000000,1.600000,1.000000,1.000000,21000.000000,0.610000,0.102000,0.200000,0.115000,0.248000,0.100000,0.200000,0.100000,0.200000,0.200000,0.200000,0.181000,0.569000,0.569000,0.000000
25%,8.100000,6.000000,7.500000,11.000000,15.000000,102000.000000,1.000000,0.350000,0.488000,0.338000,0.518000,0.100000,0.200000,0.500000,0.657000,0.591000,0.603000,0.368000,0.712000,0.833000,6.000000
50%,8.400000,10.000000,8.600000,16.000000,20.000000,150000.000000,1.000000,0.484000,0.488000,0.492000,0.518000,0.990000,1.000000,0.500000,0.657000,0.732000,0.749000,0.541000,0.801000,0.894000,14.000000
75%,8.600000,15.000000,9.700000,20.000000,20.000000,210000.000000,1.000000,0.539000,0.488000,0.565000,0.518000,0.990000,1.000000,0.750000,0.770000,0.796000,0.806000,0.676000,0.856000,0.964000,19.000000
max,10.000000,57.000000,10.000000,56.000000,82.000000,560000.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.990000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,24.000000


## Make Test Dataset for Evaluation

In [7]:
### Fill the null values with average for some columns
salaries['company_score'] = salaries['company_score'].fillna(salaries['company_score'].mean())
salaries['education_score'] = salaries['education_score'].fillna(salaries['education_score'].mean())
### Select the columns (features) for ML models
salaries_sub = salaries[['company_score','working_year','education_score','ms_counts','skill_counts','country_coef', \
                         'title_slr_coef','title_seniority_slr_coef','position_slr_coef','position_seniority_slr_coef', \
                         'employment_type_coef','employment_type_slr_coef','highest_edu_deg_lvl','edu_deg_slr_coef', \
                         'edu_degree_major_slr_coef_max','company_slr_coef','main_skill_slr_coef_avg','amount_usd']]

In [8]:
X = salaries_sub.drop('amount_usd',axis=1)
Y = salaries_sub['amount_usd']
random_state = random.randint(1,30)
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.1, random_state=random_state)

In [9]:
print("TRAIN SIZE:", x_train.shape, y_train.shape, "| TEST SIZE:", x_test.shape, y_test.shape, "| RANDOM STATE:", random_state)

TRAIN SIZE: (17598, 17) (17598,) | TEST SIZE: (1956, 17) (1956,) | RANDOM STATE: 7


## Evaluate ML Regression Models

In [10]:
def adjusted_r2_score(r2_score, n, k):
    return 1 - (1-r2_score)*(n-1)/(n-k-1)

### 1. Linear Regression

In [11]:
with open('ml_linreg_proto.pkl', 'rb') as ml_file:
    linreg = pickle.load(ml_file)
y_pred = linreg.predict(x_test)

In [12]:
print("MSE: ", mean_squared_error(y_test, y_pred))
print("RMSE:", mean_squared_error(y_test, y_pred, squared=False))
print("MAE: ", mean_absolute_error(y_test, y_pred))
print("R2 SCORE:", r2_score(y_test, y_pred))
print("ADJUSTED R2 SCORE:", adjusted_r2_score(r2_score(y_test, y_pred), len(y_test), X.shape[1]))

MSE:  2008424172.4972217
RMSE: 44815.445691159
MAE:  33856.93153866677
R2 SCORE: 0.7008167899751947
ADJUSTED R2 SCORE: 0.6981923758521701


In [13]:
s_time = time.time()
cross_val = cross_validate(linreg, X, Y, cv=5, return_train_score=True)
print("### 5-FOLD CROSS VALIDATION ###")   # Completed in less than a second
print("FITTING TIME: {} -> {:.8f}".format(cross_val['fit_time'], cross_val['fit_time'].mean()))
print("SCORING TIME: {} -> {:.8f}".format(cross_val['score_time'], cross_val['score_time'].mean()))
print("TRAIN SCORE:  {} -> {:.8f}".format(cross_val['train_score'], cross_val['train_score'].mean()))
print("TEST SCORE:   {} -> {:.8f}".format(cross_val['test_score'], cross_val['test_score'].mean()))     
print("TIME TAKEN FOR CROSS VALIDATION: {:.4f} seconds".format(time.time()-s_time))

### 5-FOLD CROSS VALIDATION ###
FITTING TIME: [0.04914784 0.01972628 0.01368356 0.01032329 0.015625  ] -> 0.02170119
SCORING TIME: [0.00399637 0.         0.01437163 0.         0.        ] -> 0.00367360
TRAIN SCORE:  [0.67881205 0.68191236 0.68064357 0.67801697 0.67771902] -> 0.67942080
TEST SCORE:   [0.68084273 0.66838669 0.6737038  0.68421857 0.68502656] -> 0.67843567
TIME TAKEN FOR CROSS VALIDATION: 0.1446 seconds


### 2. Random Forest Regression

In [14]:
with open('ml_rforest_reg_proto.pkl', 'rb') as ml_file:
    rforest = pickle.load(ml_file)
y_pred = rforest.predict(x_test)

In [15]:
print("MSE: ", mean_squared_error(y_test, y_pred))
print("RMSE:", mean_squared_error(y_test, y_pred, squared=False))
print("MAE: ", mean_absolute_error(y_test, y_pred))
print("R2 SCORE:", r2_score(y_test, y_pred))
print("ADJUSTED R2 SCORE:", adjusted_r2_score(r2_score(y_test, y_pred), len(y_test), X.shape[1]))

MSE:  143516981.95254785
RMSE: 11979.857342746109
MAE:  8226.439764826177
R2 SCORE: 0.9786211140347671
ADJUSTED R2 SCORE: 0.9784335799473528


In [16]:
s_time = time.time()
cross_val = cross_validate(rforest, X, Y, cv=5, return_train_score=True)
print("### 5-FOLD CROSS VALIDATION ###")   # Completed in ~25 seconds
print("FITTING TIME: {} -> {:.8f}".format(cross_val['fit_time'], cross_val['fit_time'].mean()))
print("SCORING TIME: {} -> {:.8f}".format(cross_val['score_time'], cross_val['score_time'].mean()))
print("TRAIN SCORE:  {} -> {:.8f}".format(cross_val['train_score'], cross_val['train_score'].mean()))
print("TEST SCORE:   {} -> {:.8f}".format(cross_val['test_score'], cross_val['test_score'].mean()))     
print("TIME TAKEN FOR CROSS VALIDATION: {:.4f} seconds".format(time.time()-s_time))

### 5-FOLD CROSS VALIDATION ###
FITTING TIME: [4.70365763 4.71571612 4.93913746 5.16886854 5.10170007] -> 4.92581596
SCORING TIME: [0.03014374 0.03184867 0.03132439 0.03115821 0.0311482 ] -> 0.03112464
TRAIN SCORE:  [0.9755138  0.97524064 0.97418532 0.97412158 0.97539652] -> 0.97489157
TEST SCORE:   [0.81800943 0.82154008 0.83264538 0.83911555 0.82866553] -> 0.82799520
TIME TAKEN FOR CROSS VALIDATION: 25.6785 seconds


### 3. Gradient Boosting Regression

In [17]:
with open('ml_gradboost_reg_proto.pkl', 'rb') as ml_file:
    gradBoost = pickle.load(ml_file)
y_pred = gradBoost.predict(x_test)

In [18]:
print("MSE: ", mean_squared_error(y_test, y_pred))
print("RMSE:", mean_squared_error(y_test, y_pred, squared=False))
print("MAE: ", mean_absolute_error(y_test, y_pred))
print("R2 SCORE:", r2_score(y_test, y_pred))
print("ADJUSTED R2 SCORE:", adjusted_r2_score(r2_score(y_test, y_pred), len(y_test), X.shape[1]))

MSE:  1360600675.712339
RMSE: 36886.32098369718
MAE:  27160.97472862527
R2 SCORE: 0.7973192698555316
ADJUSTED R2 SCORE: 0.7955413687139135


In [19]:
s_time = time.time()
cross_val = cross_validate(gradBoost, X, Y, cv=5, return_train_score=True)
print("### 5-FOLD CROSS VALIDATION ###")   # Completed in less than 10 seconds
print("FITTING TIME: {} -> {:.8f}".format(cross_val['fit_time'], cross_val['fit_time'].mean()))
print("SCORING TIME: {} -> {:.8f}".format(cross_val['score_time'], cross_val['score_time'].mean()))
print("TRAIN SCORE:  {} -> {:.8f}".format(cross_val['train_score'], cross_val['train_score'].mean()))
print("TEST SCORE:   {} -> {:.8f}".format(cross_val['test_score'], cross_val['test_score'].mean()))     
print("TIME TAKEN FOR CROSS VALIDATION: {:.4f} seconds".format(time.time()-s_time))

### 5-FOLD CROSS VALIDATION ###
FITTING TIME: [1.68831348 1.69493318 1.69577098 1.63662195 1.6729188 ] -> 1.67771168
SCORING TIME: [0.01717186 0.01678395 0.         0.00557709 0.01453328] -> 0.01081324
TRAIN SCORE:  [0.78162969 0.78076233 0.7803484  0.77732798 0.78034267] -> 0.78008221
TEST SCORE:   [0.76422141 0.76114982 0.77206574 0.77708795 0.77067819] -> 0.76904062
TIME TAKEN FOR CROSS VALIDATION: 8.5473 seconds


### 4. XGBoost Regression

In [20]:
with open('ml_xgboost_reg_proto.pkl', 'rb') as ml_file:
    xgb = pickle.load(ml_file)
y_pred = xgb.predict(x_test)

In [21]:
print("MSE: ", mean_squared_error(y_test, y_pred))
print("RMSE:", mean_squared_error(y_test, y_pred, squared=False))
print("MAE: ", mean_absolute_error(y_test, y_pred))
print("R2 SCORE:", r2_score(y_test, y_pred))
print("ADJUSTED R2 SCORE:", adjusted_r2_score(r2_score(y_test, y_pred), len(y_test), X.shape[1]))

MSE:  567715290.740954
RMSE: 23826.776759372093
MAE:  17368.81915819913
R2 SCORE: 0.9154307713530174
ADJUSTED R2 SCORE: 0.9146889360140088


In [22]:
s_time = time.time()
cross_val = cross_validate(xgb, X, Y, cv=5, return_train_score=True)
print("### 5-FOLD CROSS VALIDATION ###")   # Completed in less than a second
print("FITTING TIME: {} -> {:.8f}".format(cross_val['fit_time'], cross_val['fit_time'].mean()))
print("SCORING TIME: {} -> {:.8f}".format(cross_val['score_time'], cross_val['score_time'].mean()))
print("TRAIN SCORE:  {} -> {:.8f}".format(cross_val['train_score'], cross_val['train_score'].mean()))
print("TEST SCORE:   {} -> {:.8f}".format(cross_val['test_score'], cross_val['test_score'].mean()))     
print("TIME TAKEN FOR CROSS VALIDATION: {:.4f} seconds".format(time.time()-s_time))

### 5-FOLD CROSS VALIDATION ###
FITTING TIME: [0.12523842 0.12896729 0.11745548 0.14108133 0.11512446] -> 0.12557340
SCORING TIME: [0.00000000e+00 0.00000000e+00 0.00000000e+00 6.24656677e-05
 1.10061169e-02] -> 0.00221372
TRAIN SCORE:  [0.9156003  0.9134585  0.91494021 0.91411754 0.91180045] -> 0.91398340
TEST SCORE:   [0.81600683 0.81744898 0.82857644 0.83281611 0.82716802] -> 0.82440328
TIME TAKEN FOR CROSS VALIDATION: 0.7183 seconds


## Assign Topic Number for Single Dataset Entry

In [23]:
### Process the null titles and companies to become empty strings
salaries['title'] = salaries['title'].apply(lambda x: '' if type(x) == float else x)
salaries['company'] = salaries['company'].apply(lambda x: '' if type(x) == float else x)

In [24]:
### Load the dictionary and LDA model
dictionary_file = datapath('dict_tm25')
dictionary = gensim.corpora.Dictionary.load(dictionary_file)
lda_model_file = datapath('lda_model_tm25')
lda_model = gensim.models.LdaModel.load(lda_model_file)

In [25]:
### Select a row randomly from the dataset and show its textual values
rand_row = random.randint(0,len(salaries)-1)

print("#####  ROW {}  #####".format(rand_row))
print(salaries.loc[rand_row,['id','found_country','title','position','employment_type','company','edu_degrees', \
                             'edu_degrees_major','main_skills','skills']])

#####  ROW 16459  #####
id                                d75d29c4-055e-4ecf-b8d2-fd4a12aa2578
found_country                                            United States
title                                        Data Analyst at Microsoft
position                                                  Data Analyst
employment_type                                               Contract
company                                                     Microsoft 
edu_degrees                                      MASTERS,UNDERGRADUATE
edu_degrees_major    Bachelor's degree, Business Administration and...
main_skills          Data Analysis, Data Engineering, Marketing Res...
skills               data analysis, microsoft excel, microsoft offi...
Name: 16459, dtype: object


In [26]:
### Convert all textual values into list vectors
### All characters must be lowercase and several specific characters and substring should be converted into meaningful ones
### Note that the textual values are currently limited to title, main skills and skills
single_row = salaries.loc[rand_row]
single_row_text = (single_row['title'] + ' ' + single_row['main_skills'] + ' ' + single_row['skills']).lower()
s_time = time.time()
manipulated_text = manipulate_text(single_row_text)
translated_text = translate_to_english(manipulated_text)
single_row_text_processed = translated_text
single_row_text_vector = single_row_text_processed.split(' ')
print("### ROW {} ###\n{}".format(rand_row, single_row_text_vector))

### ROW 16459 ###
['data', 'analyst', 'microsoft', 'data', 'analysis', 'data', 'engineering', 'marketing', 'research', 'microsoft', 'excel', 'microsoft', 'office', 'microsoft', 'sql', 'server', 'microsoft', 'word', 'mongodb', 'oracle', 'sql', 'developer', 'powerpoint', 'programming', 'python', 'r', 'research', 'sql', 'sql', 'scripting', 'tableau', 'data', 'analysis', 'microsoft', 'excel', 'microsoft', 'office', 'microsoft', 'powerpoint', 'microsoft', 'sql', 'server', 'microsoft', 'word', 'mongodb', 'oracle', 'sql', 'developer', 'python', 'python', 'programming', 'language', 'r', 'r', 'studio', 'research', 'sql', 'tableau']


In [27]:
### Get Bag of Words vector from the dictionary
bow_vector = dictionary.doc2bow(single_row_text_vector)
print("### ROW {} - BoW VECTOR ###\n{}".format(rand_row, bow_vector))

### ROW 16459 - BoW VECTOR ###
[(6, 4), (8, 2), (10, 1), (16, 10), (18, 2), (19, 2), (21, 2), (22, 3), (24, 1), (27, 7), (41, 1), (55, 2), (123, 2), (128, 3), (156, 2), (164, 1), (169, 1), (174, 2), (175, 3), (183, 2), (394, 2), (598, 1), (608, 2)]


In [28]:
### Get the topic scores in descending order by running the LDA model
scores = sorted(lda_model[bow_vector], key=lambda tup: -1*tup[1])
print("### ROW {} - LDA TOPIC SCORES ###\n{}".format(rand_row, scores))
assigned_topic = scores[0][0]
print("ASSIGNED TOPIC -->", assigned_topic)

### ROW 16459 - LDA TOPIC SCORES ###
[(24, 0.34621093), (14, 0.34577268), (11, 0.25648803), (23, 0.026513785), (5, 0.010639805)]
ASSIGNED TOPIC --> 24


In [29]:
### Get the related words for each available topics
topic_words = {}
for index, score in scores:
    topic_words['TOPIC '+str(index)] = lda_model.print_topic(index, 6)
topic_words

{'TOPIC 24': '0.291*"microsoft" + 0.102*"office" + 0.090*"excel" + 0.071*"powerpoint" + 0.068*"word" + 0.051*"research"',
 'TOPIC 14': '0.197*"data" + 0.088*"analysis" + 0.060*"sql" + 0.041*"analytics" + 0.030*"python" + 0.029*"programming"',
 'TOPIC 11': '0.140*"sql" + 0.056*"oracle" + 0.049*"scripting" + 0.045*"server" + 0.041*"data" + 0.038*"microsoft"',
 'TOPIC 23': '0.105*"data" + 0.065*"learning" + 0.060*"machine" + 0.043*"python" + 0.037*"r" + 0.036*"sql"',
 'TOPIC 5': '0.132*"html" + 0.103*"css" + 0.074*"javascript" + 0.054*"development" + 0.041*"web" + 0.028*"nodejs"'}

## Predict Salary from Single Dataset Entry

In [30]:
### With the same random row from the dataset, get the features about numerical values and salary coefficients
# rand_row = random.randint(0,len(salaries)-1)
print("#####  ROW {}  #####".format(rand_row))
print(salaries_sub.iloc[rand_row])

#####  ROW 16459  #####
company_score                         8.300
working_year                          3.000
education_score                       8.500
ms_counts                            17.000
skill_counts                         15.000
country_coef                          1.000
title_slr_coef                        0.359
title_seniority_slr_coef              0.488
position_slr_coef                     0.360
position_seniority_slr_coef           0.518
employment_type_coef                  0.500
employment_type_slr_coef              0.809
highest_edu_deg_lvl                   0.750
edu_deg_slr_coef                      0.770
edu_degree_major_slr_coef_max         0.732
company_slr_coef                      0.676
main_skill_slr_coef_avg               0.800
amount_usd                       113000.000
Name: 16459, dtype: float64


In [31]:
### Get the numerical values from the row and make predictions with all ML models
single_row_numeric = salaries_sub.iloc[rand_row].drop('amount_usd').values.reshape(1,-1)
print("ROW {} --> {}".format(rand_row, single_row_numeric))
actual = salaries_sub.iloc[rand_row,-1]
pred1 = linreg.predict(single_row_numeric)[0]
pred2 = rforest.predict(single_row_numeric)[0]
pred3 = gradBoost.predict(single_row_numeric)[0]
pred4 = xgb.predict(single_row_numeric)[0]
pred_c1 = np.array([pred1,pred2,pred3,pred4]).mean()
pred_c2 = np.array([pred2,pred3,pred4]).mean()
print("#### PREDICTION RESULTS ####")
print("ACTUAL            : {:14.6f}".format(actual))
print("-"*55)
print("LINEAR REG.       : {:14.6f}    ({:14.6f})".format(pred1, pred1-actual))
print("RANDOM FOREST REG.: {:14.6f}    ({:14.6f})".format(pred2, pred2-actual))
print("GRAD BOOST REG.   : {:14.6f}    ({:14.6f})".format(pred3, pred3-actual))
print("XGBOOST REG.      : {:14.6f}    ({:14.6f})".format(pred4, pred4-actual))
print("-"*55)
print("COMBINED ALL      : {:14.6f}    ({:14.6f})".format(pred_c1, pred_c1-actual))
print("COMBINED 2/3/4    : {:14.6f}    ({:14.6f})".format(pred_c2, pred_c2-actual))

ROW 16459 --> [[ 8.3    3.     8.5   17.    15.     1.     0.359  0.488  0.36   0.518
   0.5    0.809  0.75   0.77   0.732  0.676  0.8  ]]
#### PREDICTION RESULTS ####
ACTUAL            :  113000.000000
-------------------------------------------------------
LINEAR REG.       :  143437.859678    (  30437.859678)
RANDOM FOREST REG.:  114920.000000    (   1920.000000)
GRAD BOOST REG.   :  130688.385856    (  17688.385856)
XGBOOST REG.      :  114860.437500    (   1860.437500)
-------------------------------------------------------
COMBINED ALL      :  125976.670758    (  12976.670758)
COMBINED 2/3/4    :  120156.274452    (   7156.274452)


## Computations for Previous Heuristic Model

In [39]:
### Get all working years and salaries that has the same assigned topic
years_salaries_tpc = salaries[salaries['topic_model_no'] == assigned_topic][['working_year','amount_usd']].reset_index()
# Take each necesary info to be used in the computations below
title = single_row['title']
country = single_row['found_country']
country_coef = single_row['country_coef']
wr_yr = single_row['working_year']
edu_score = single_row['education_score']
comp_score = single_row['company_score']
sv_score = 'LOW'
pep_man = 0

print("#### INFO ABOUT THE SELECTED ROW ({}) ####".format(rand_row))
print("TITLE:", title)
print("COUNTRY:", country)
print("COUNTRY COEF:", country_coef)
print("WORIKNG YEAR:", wr_yr)
print("EDUCATION SCORE:", edu_score)
print("COMPANY SCORE:", comp_score)
print("SUPERVISION SCORE:", sv_score, "(MANUALLY ENTERED)")
print("# OF PEOPLE MANAGED:", pep_man, "(MANUALLY ENTERED)")
print("ASSIGNED TOPIC: {} ({} TOTAL SALARIES WITH THIS TOPIC)\n".format(assigned_topic, years_salaries_tpc.shape[0]))

### Inspect the candidate's title and determine the title coefficient (currently disabled)
if 'director' in title:           title_coef = 2.4
elif 'senior manager' in title:   title_coef = 1.8
elif 'manager' in title:          title_coef = 1.6
elif 'lead' in title:             title_coef = 1.3
else:   title_coef = 1.0
print("TITLE COEF: {}".format(title_coef))
### Inspect the candidate's supervision score and determine the supervision coefficient
if sv_score == 10 or sv_score == 'HIGH':      supervision_coef = 1.0   # 2.2
elif sv_score == 6 or sv_score == 'MEDIUM':   supervision_coef = 1.0   # 1.6
elif sv_score == 2 or sv_score == 'LOW':      supervision_coef = 1.0
else:    supervision_coef = 1.0
print("SUPERVISION COEF: {}".format(supervision_coef))
# ##Inspect the number of people being managed and determine the people management coefficient
if pep_man >= 1001:   pep_man_coef = 5.5
elif pep_man >= 501 and pep_man <= 1000:   pep_man_coef = 4.8
elif pep_man >= 101 and pep_man <= 500:    pep_man_coef = 3.6
elif pep_man >= 26 and pep_man <= 100:     pep_man_coef = 2.8
elif pep_man >= 11 and pep_man <= 25:      pep_man_coef = 1.6
elif pep_man >= 1 and pep_man <= 10:       pep_man_coef = 1.3
else:   pep_man_coef = 1.0
print("PEOPLE MANAGEMENT COEF: {}".format(pep_man_coef))
### Inspect the candidate's education score and determine the education coefficient
if edu_score >= 9.5:   edu_coef = 2.0
elif edu_score >= 9.0 and edu_score <= 9.4:   edu_coef = 1.7
elif edu_score >= 8.0 and edu_score <= 8.9:   edu_coef = 1.4
elif edu_score >= 7.0 and edu_score <= 7.9:   edu_coef = 1.2
else:   edu_coef = 1.0
print("EDUCATION COEF: {}".format(edu_coef))
### Inspect the candidate's last company score and determine the company coefficient
if comp_score >= 9.0:   comp_coef = 2.5
elif comp_score >= 8.0 and comp_score <= 8.9:   comp_coef = 2.0
else:   comp_coef = 1.0
print("COMPANY COEF: {}".format(comp_coef))
### From these determined education and company coefs and the candidate's working year, get the weighted coefficient
if wr_yr < 2:   edu_weight = 0.5;   comp_weight = 0.5
elif wr_yr >= 2 and wr_yr <= 5:   edu_weight = 0.3;   comp_weight = 0.7
elif wr_yr >= 6 and wr_yr <= 10:   edu_weight = 0.2;   comp_weight = 0.8
elif wr_yr >= 11:   edu_weight = 0.1;   comp_weight = 0.9
weighted_coef = edu_weight * edu_coef + comp_weight * comp_coef
print("WEIGHTED COEF: ({:.1f} * {:.1f}) + ({:.1f} * {:.1f}) = {:.3f}  (WORKING YEAR = {})".format(
    edu_weight, edu_coef, comp_weight, comp_coef, weighted_coef, wr_yr))
print("COUNTRY COEF: {:.3f}".format(country_coef))

### After all these calculations, obtain the salary coefficient to be applied after the prediction
### However, there must be valid country coefficient, otherwise it will not continue!
if country_coef == None:
    print("WARNING! COUNTRY COEFFICIENT DOES NOT EXIST!")
elif pep_man_coef != 1.0:   
    salary_coef = pep_man_coef * weighted_coef * country_coef
    print(">>> SALARY COEFFICIENT: {:.2f} * {:.2f} * {:.2f} = {:.3f}".format(pep_man_coef, weighted_coef, country_coef, salary_coef))
else:   
    salary_coef = supervision_coef * weighted_coef * country_coef
    print(">>> SALARY COEFFICIENT: {:.2f} * {:.2f} * {:.2f} = {:.3f}".format(supervision_coef, weighted_coef, country_coef, salary_coef))

#### INFO ABOUT THE SELECTED ROW (16459) ####
TITLE: Data Analyst at Microsoft
COUNTRY: United States
COUNTRY COEF: 1.0
WORIKNG YEAR: 3
EDUCATION SCORE: 8.5
COMPANY SCORE: 8.3
SUPERVISION SCORE: LOW (MANUALLY ENTERED)
# OF PEOPLE MANAGED: 0 (MANUALLY ENTERED)
ASSIGNED TOPIC: 24 (1416 TOTAL SALARIES WITH THIS TOPIC)

TITLE COEF: 1.0
SUPERVISION COEF: 1.0
PEOPLE MANAGEMENT COEF: 1.0
EDUCATION COEF: 1.4
COMPANY COEF: 2.0
WEIGHTED COEF: (0.3 * 1.4) + (0.7 * 2.0) = 1.820  (WORKING YEAR = 3)
COUNTRY COEF: 1.000
>>> SALARY COEFFICIENT: 1.00 * 1.82 * 1.00 = 1.820


In [33]:
### Determine the categories of working years and assign the min-max experience years
years_salaries_count = {};   years_salaries = {}
if wr_yr >= 15:   
    years_range = ['10-14', '15+'];            minExp = 10;    maxExp = 50
elif wr_yr >= 10 and wr_yr <= 14:  
    years_range = ['7-9', '10-14', '15+'];     minExp = 7;    maxExp = 50
elif wr_yr >= 7 and wr_yr <= 9:   
    years_range = ['4-6', '7-9', '10-14'];     minExp = 4;    maxExp = 14
elif wr_yr >= 4 and wr_yr <= 6:     
    years_range = ['1-3', '4-6', '7-9'];       minExp = 1;    maxExp = 9
elif wr_yr >= 1 and wr_yr <= 3:
    years_range = ['0', '1-3', '4-6'];         minExp = 0;    maxExp = 6
else:    
    years_range = ['0', '1-3'];                minExp = 0;    maxExp = 3
print("MIN-MAX YEAR EXPERIENCE CATEGORIES: {}".format(years_range))
print("MIN-MAX YEAR EXPERIENCE RANGE: {}-{}".format(minExp, maxExp))

### Determine the salary interval thresholds w.r.t. the country coefficient
salary_intrv_thr = [int(country_coef * intrv) for intrv in [100000, 140000, 180000, 220000, 260000, 300000, 340000]]
# print("SALARY INTERVAL THRESHOLDS:", salary_intrv_thr)
# Construct their interval labels as yearly
salary_intrv_label_y = []
for i in range(len(salary_intrv_thr)):
    if i == 0:
        salary_intrv_label_y.append('<'+str(salary_intrv_thr[i] // 1000)+'K')
        salary_intrv_label_y.append(str(salary_intrv_thr[i] // 1000)+'K-'+str(salary_intrv_thr[i+1] // 1000)+'K')
    elif i == len(salary_intrv_thr)-1:
        salary_intrv_label_y.append('>='+str(salary_intrv_thr[i] // 1000)+'K')
    else:
        salary_intrv_label_y.append(str(salary_intrv_thr[i] // 1000)+'K-'+str(salary_intrv_thr[i+1] // 1000)+'K')
print("SALARY INTERVAL LABELS (YEAR):", salary_intrv_label_y)
# Do the same intervals as monthly
salary_intrv_label_m = []
for i in range(len(salary_intrv_thr)):
    if i == 0:
        salary_intrv_label_m.append('<'+str(salary_intrv_thr[i] // 1000 // 12)+'K')
        salary_intrv_label_m.append(str(salary_intrv_thr[i] // 1000 // 12)+'K-'+str(salary_intrv_thr[i+1] // 1000 // 12)+'K')
    elif i == len(salary_intrv_thr)-1:
        salary_intrv_label_m.append('>='+str(salary_intrv_thr[i] // 1000 // 12)+'K')
    else:
        salary_intrv_label_m.append(str(salary_intrv_thr[i] // 1000 // 12)+'K-'+str(salary_intrv_thr[i+1] // 1000 // 12)+'K')
print("SALARY INTERVAL LABELS (MONTH):", salary_intrv_label_m, "\n")

### Place all these salaries of the subset into the appropriate lists w.r.t. their categories of working years & salary intervals
### This is done for individual working years category and the combination of all
for year in years_range:
    years_salaries_count[year] = [0 for _ in range(8)]
    years_salaries[year] = [[] for _ in range(8)]
years_salaries_comb = [[] for _ in range(8)]
years_salaries_whole = []
for i in range(len(years_salaries_tpc)):
    year = years_salaries_tpc.loc[i, 'working_year']
    salary = years_salaries_tpc.loc[i, 'amount_usd']
    # Determine the condition of working years for correct selection
    # Pay attention to both the candidate's and the related row's working year together
    # If the related row's working year value is not in one of the categories, skip to the next one
    if wr_yr >= 15 and year >= 10:
        year_cond = '10-14' if year >= 10 and year <= 14 else '15+'
    elif (wr_yr >= 10 and wr_yr <= 14) and year >= 7:
        year_cond = '7-9' if year >= 7 and year <= 9 else '10-14' if year >= 10 and year <= 14 else '15+'
    elif (wr_yr >= 7 and wr_yr <= 9) and (year >= 4 and year <= 14):
        year_cond = '4-6' if year >= 4 and year <= 6 else '7-9' if year >= 7 and year <= 9 else '10-14' 
    elif (wr_yr >= 4 and wr_yr <= 6) and (year >= 1 and year <= 9):
        year_cond = '1-3' if year >= 1 and year <= 3 else '4-6' if year >= 4 and year <= 6 else '7-9'
    elif (wr_yr >= 1 and wr_yr <= 3) and year <= 6:
        year_cond = '0' if year == 0 else '1-3' if year >= 1 and year <= 3 else '4-6'
    elif wr_yr == 0 and year <= 3:
        year_cond = '0' if year == 0 else '1-3'
    else:
        continue
    salaries_cond = 0 if salary < 100000 else 1 if salary >= 100000 and salary < 140000 \
        else 2 if salary >= 140000 and salary < 180000 else 3 if salary >= 180000 and salary < 220000 \
        else 4 if salary >= 220000 and salary < 260000 else 5 if salary >= 260000 and salary < 300000 \
        else 6 if salary >= 300000 and salary < 340000 else 7
    # Count the salary according to the conditions set above and append it to the appropriate salary interval list
    years_salaries_count[year_cond][salaries_cond] += 1
    years_salaries[year_cond][salaries_cond].append(salary)
    years_salaries_comb[salaries_cond].append(salary)
    years_salaries_whole.append(salary)

### Make a dataframe of the counted salaries w.r.t. working year categories and salary intervals
years_salaries_df = pd.DataFrame(years_salaries_count, index=salary_intrv_label_y)
print(years_salaries_df)
# Get the total number of salaries included in the counts and lists (also get results by years and intervals)
total_salaries = years_salaries_df.sum().sum()
total_salaries_years = years_salaries_df.sum().values
total_salaries_intrv = years_salaries_df.sum(axis=1).values
print("{} SALARIES AVAILABLE IN THIS RANGE\nBY YEARS RANGE: {}\nBY INTERVALS:   {}\n".format(
      total_salaries, total_salaries_years, total_salaries_intrv))
# Get the percentage distribution of all defined salary intervals for all included salaries
if total_salaries != 0:
    print("SALARY INTERVALS PERCENTAGE:", np.around(years_salaries_df.sum(axis=1).values / total_salaries * 100, 3).tolist())

### Show the salaries appended to each interval salary list
### And calculate the percentage distribution for each working year category, then reveal them to the output
years_salaries_prob = [];   years_salaries_prob_comb = [];  i = 0
for year in years_range:
    # Be aware that the salary list might be too long!
    # print("SALARIES ({}): {}".format(year, years_salaries[year]))
    if sum(years_salaries_count[year]) != 0:
        years_salaries_prob.append([round(val / sum(years_salaries_count[year]) * 100, 3) 
                                    for val in years_salaries_count[year]])
    else:
        years_salaries_prob.append([0.0 for _ in range(8)])  
    print("PERCENTAGE OF {} YEARS: {}".format(year, years_salaries_prob[i]))
    i += 1
# Do the same thing above for the combined salaries of each interval regardless of working year categories
# print("SALARIES COMBINED: {}".format(years_salaries_comb))  # Be aware that the salary list might be too long!
for i in range(len(years_salaries_comb)):
    years_salaries_prob_comb.append(round(len(years_salaries_comb[i]) / years_salaries_df.sum().sum() * 100, 3)
                                    if len(years_salaries_comb[i]) != 0 else 0.0)
print("PERCENTAGE COMBINED: {}".format(years_salaries_prob_comb))

### Calculate the predicted salary for each working year category (there should be two or three years range)
### However, any working year category that does not have any salary data will not be included
years_salaries_calc = [];  j = 0
for year in years_range:
    calc_value = 0.0
    for i in range(len(years_salaries[year])):
        if len(years_salaries[year][i]) == 0:
            continue
        # print(years_salaries[year][i])
        # Value is added by getting the average of salaries in the interval, multiplied by its percentage in the dist.
        calc_value += np.array(years_salaries[year][i]).mean() * (years_salaries_prob[j][i] / 100)
    if calc_value != 0.0:
        years_salaries_calc.append(round(calc_value, 4))
    j += 1
print("\nPREDICTED SALARIES OF EACH YEAR CATEGORY")
for x in range(len(years_range)):
    print("{:7} --> {}".format(years_range[x], years_salaries_calc[x]))
years_salaries_avg = np.array(years_salaries_calc).mean()
print("AVERAGE --> {:.4f}".format(years_salaries_avg))
# The same operation is also carried out with the combined variant
years_salaries_calc_comb = 0.0
for i in range(len(years_salaries_comb)):
    if len(years_salaries_comb[i]) == 0:
        continue
    years_salaries_calc_comb += np.array(years_salaries_comb[i]).mean() * (years_salaries_prob_comb[i] / 100)
print("PREDICTED SALARY (COMBINED):", round(years_salaries_calc_comb, 4))

print("\n{} SALARIES WERE USED FOR PREDICTION AND {:.3f} SALARY COEF WAS APPLIED AFTERWARDS".format(total_salaries, salary_coef))
print("##### MIN-MAX SALARY RESULTS #####")
### OPTION 1: Get the minimum and maximum value from the predicted salaries of each year category
print("OPTION 1 - YEAR CATEGORY MIN-MAX >>  {:10.3f} - {:10.3f}  ({:10.3f} - {:10.3f})".format(
    round(min(years_salaries_calc),-2), round(max(years_salaries_calc),-2),
    round(min(years_salaries_calc) * salary_coef,-2), round(max(years_salaries_calc) * salary_coef,-2)))
### OPTION 2: Find the average of the predicted salaries of each year category
###           And get 90% and 110% of the average to assess them as min-max values 
print("OPTION 2 - YEAR CATEGORY AVERAGE >>  {:10.3f} - {:10.3f}  ({:10.3f} - {:10.3f})".format(
    round(years_salaries_avg * 0.9,-2), round(years_salaries_avg * 1.1,-2),
    round(years_salaries_avg * 0.9 * salary_coef,-2), round(years_salaries_avg * 1.1 * salary_coef,-2)))
### OPTION 3: Get 90% and 110% of the combined variant of predicted salary to assess them as min-max values
print("OPTION 3 - YEAR CATEGORY COMBINED AVERAGE >>  {:10.3f} - {:10.3f}  ({:10.3f} - {:10.3f})".format(
    round(years_salaries_calc_comb * 0.9,-2), round(years_salaries_calc_comb * 1.1,-2),
    round(years_salaries_calc_comb * 0.9 * salary_coef,-2), round(years_salaries_calc_comb * 1.1 * salary_coef,-2)))

### Determine the median value of salaries with three different options applied below
### The numbers in parentheses indicate the total number of salaries included for finding the median
print("\n##### MEDIAN RESULTS #####")
# OPTION 1: All salaries from the assigned topic
print("OPTION 1 - ALL SALARIES IN CHOSEN TOPIC ({}) >>  {:.3f}  ({:.3f})".format(
    len(years_salaries_tpc), round(np.median(years_salaries_tpc['amount_usd']),-2),
    round(np.median(years_salaries_tpc['amount_usd']) * salary_coef,-2)))
# OPTION 2: All salaries included in working year categories
ys_whole = np.array(years_salaries_whole)
print("OPTION 2 - ALL SALARIES IN WORKING YEARS RANGE ({}) >>  {:.3f}  ({:.3f})".format(
    len(ys_whole), round(np.median(ys_whole),-2), round(np.median(ys_whole) * salary_coef,-2)))
# OPTION 3: All salaries that are within the min-max range of predicted salary
# NOTE: Salary prediction result OPTION 2 is considered for the range
ys_whole_range = ys_whole[(ys_whole > years_salaries_avg * 0.9) & (ys_whole < years_salaries_avg * 1.1)]
print("OPTION 3 - ALL SALARIES IN PREDICTED MIN-MAX ({}) >>  {:.3f}  ({:.3f})".format(
    len(ys_whole_range), round(np.median(ys_whole_range),-2), round(np.median(ys_whole_range) * salary_coef,-2)))
# print(np.sort(ys_whole_range))

# Perform the prediction of yearly salaries by over 40 different countries at once
# Note that the 2ND OPTION of min-max salary results are being used
countries_for_salary_calc = ['Argentina', 'Australia', 'Austria', 'Bangladesh', 'Belarus', 'Belgium', 'Brazil', 'Canada', 
    'China', 'Denmark', 'Egypt', 'Estonia', 'Finland', 'France', 'Germany', 'Iceland', 'India', 'Indonesia', 'Ireland', 
    'Israel', 'Italy', 'Japan', 'Luxembourg', 'Mexico', 'Netherlands', 'New Zealand', 'Norway', 'Pakistan', 'Philippines', 
    'Poland', 'Romania', 'Russia', 'Serbia', 'Singapore', 'Spain', 'South Korea', 'Sweden', 'Switzerland', 'Turkey', 
    'Ukraine', 'United Kingdom', 'United States', 'Vietnam']
print("\n##### SALARIES BY COUNTRIES #####")
years_salaries_avg = np.array(years_salaries_calc).mean()
countries_salaries = {};   c = 0
for country_name in countries_for_salary_calc:
    coef = countries_coef[countries_coef['en_name'] == country_name]['salary_coefficient'].values[0]
    if pep_man_coef != 1.0:
        cntr_slr = round(years_salaries_avg * pep_man_coef * weighted_coef * coef,-2)
    else:
        cntr_slr = round(years_salaries_avg * supervision_coef * weighted_coef * coef,-2)
    countries_salaries[country_name] = cntr_slr
for country_name in countries_salaries.items():
    c += 1
    if c % 3 == 0:   print("{:15} -> {:11.3f}".format(country_name[0], country_name[1]))
    else:   print("{:15} -> {:11.3f}".format(country_name[0], country_name[1]), end=" | ")

MIN-MAX YEAR EXPERIENCE CATEGORIES: ['0', '1-3', '4-6']
MIN-MAX YEAR EXPERIENCE RANGE: 0-6
SALARY INTERVAL LABELS (YEAR): ['<100K', '100K-140K', '140K-180K', '180K-220K', '220K-260K', '260K-300K', '300K-340K', '>=340K']
SALARY INTERVAL LABELS (MONTH): ['<8K', '8K-11K', '11K-15K', '15K-18K', '18K-21K', '21K-25K', '25K-28K', '>=28K'] 

           0  1-3  4-6
<100K      1   32   99
100K-140K  0   28   85
140K-180K  0    8   64
180K-220K  0   10   33
220K-260K  0    5   22
260K-300K  0    1   16
300K-340K  0    1    4
>=340K     0    0    2
411 SALARIES AVAILABLE IN THIS RANGE
BY YEARS RANGE: [  1  85 325]
BY INTERVALS:   [132 113  72  43  27  17   5   2]

SALARY INTERVALS PERCENTAGE: [32.117, 27.494, 17.518, 10.462, 6.569, 4.136, 1.217, 0.487]
PERCENTAGE OF 0 YEARS: [100.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
PERCENTAGE OF 1-3 YEARS: [37.647, 32.941, 9.412, 11.765, 5.882, 1.176, 1.176, 0.0]
PERCENTAGE OF 4-6 YEARS: [30.462, 26.154, 19.692, 10.154, 6.769, 4.923, 1.231, 0.615]
PERCENTAGE COM

In [34]:
print("#### REMEMBER THE ML MODEL'S SALARY PREDICTION RESULTS ####")
print("ACTUAL            : {:11.3f}".format(actual))
print("-"*75)
print("LINEAR REG.       : {:11.3f}   ({:11.3f})   ({:.3f} - {:.3f})".format(
    pred1, pred1 * salary_coef, round(pred1 * salary_coef * 0.9,-2), round(pred1 * salary_coef * 1.1,-2)))
print("RANDOM FOREST REG.: {:11.3f}   ({:11.3f})   ({:.3f} - {:.3f})".format(
    pred2, pred2 * salary_coef, round(pred2 * salary_coef * 0.9,-2), round(pred2 * salary_coef * 1.1,-2)))
print("GRAD BOOST REG.   : {:11.3f}   ({:11.3f})   ({:.3f} - {:.3f})".format(
    pred3, pred3 * salary_coef, round(pred3 * salary_coef * 0.9,-2), round(pred3 * salary_coef * 1.1,-2)))
print("XGBOOST REG.      : {:11.3f}   ({:11.3f})   ({:.3f} - {:.3f})".format(
    pred4, pred4 * salary_coef, round(pred4 * salary_coef * 0.9,-2), round(pred4 * salary_coef * 1.1,-2)))
print("-"*75)
print("COMBINED ALL      : {:11.3f}   ({:11.3f})   ({:.3f} - {:.3f})".format(
    pred_c1, pred_c1 * salary_coef, round(pred_c1 * salary_coef * 0.9,-2), round(pred_c1 * salary_coef * 1.1,-2)))
print("COMBINED 2/3/4    : {:11.3f}   ({:11.3f})   ({:.3f} - {:.3f})".format(
    pred_c2, pred_c2 * salary_coef, round(pred_c2 * salary_coef * 0.9,-2), round(pred_c2 * salary_coef * 1.1,-2)))

#### REMEMBER THE ML MODEL'S SALARY PREDICTION RESULTS ####
ACTUAL            :  113000.000
---------------------------------------------------------------------------
LINEAR REG.       :  143437.860   ( 261056.905)   (235000.000 - 287200.000)
RANDOM FOREST REG.:  114920.000   ( 209154.400)   (188200.000 - 230100.000)
GRAD BOOST REG.   :  130688.386   ( 237852.862)   (214100.000 - 261600.000)
XGBOOST REG.      :  114860.438   ( 209045.996)   (188100.000 - 230000.000)
---------------------------------------------------------------------------
COMBINED ALL      :  125976.671   ( 229277.541)   (206300.000 - 252200.000)
COMBINED 2/3/4    :  120156.274   ( 218684.420)   (196800.000 - 240600.000)


## Predict Salary with Custom Row

In [35]:
### Remember the order:
# company_score,        working_year,             education_score,          ms_counts,         skill_counts, 
# country_coef,         title_slr_coef,           title_seniority_slr_coef, position_slr_coef, position_seniority_slr_coef, 
# employment_type_coef, employment_type_slr_coef, highest_edu_deg_lvl,      edu_deg_slr_coef,  edu_degree_major_slr_coef_max, 
# company_slr_coef,     main_skill_slr_coef_avg
custom_row = np.array([8.300, 19.000, 8.600, 18.000, 32.000, 
                       0.820, 0.571,  0.588,  0.623,  0.518, 
                       0.800, 0.675,  0.690,  0.787,  0.632, 
                       0.505, 0.663]).reshape(1,-1)
print("CUSTOM ROW --> {}".format(custom_row))
pred1_custom = linreg.predict(custom_row)[0]
pred2_custom = rforest.predict(custom_row)[0]
pred3_custom = gradBoost.predict(custom_row)[0]
pred4_custom = xgb.predict(custom_row)[0]
pred_c1_custom = np.array([pred1,pred2,pred3,pred4]).mean()
pred_c2_custom = np.array([pred2,pred3,pred4]).mean()
print("#### PREDICTION RESULTS ####")
print("LINEAR REG.       : {:14.6f}".format(pred1_custom))
print("RANDOM FOREST REG.: {:14.6f}".format(pred2_custom))
print("GRAD BOOST REG.   : {:14.6f}".format(pred3_custom))
print("XGBOOST REG.      : {:14.6f}".format(pred4_custom))
print("-"*35)
print("COMBINED ALL      : {:14.6f}".format(pred_c1_custom))
print("COMBINED 2/3/4    : {:14.6f}".format(pred_c2_custom))

CUSTOM ROW --> [[ 8.3   19.     8.6   18.    32.     0.82   0.571  0.588  0.623  0.518
   0.8    0.675  0.69   0.787  0.632  0.505  0.663]]
#### PREDICTION RESULTS ####
LINEAR REG.       :  181826.265201
RANDOM FOREST REG.:  210700.000000
GRAD BOOST REG.   :  183667.739749
XGBOOST REG.      :  115835.664062
-----------------------------------
COMBINED ALL      :  125976.670758
COMBINED 2/3/4    :  120156.274452
